In [2]:
import pyodbc
import pandas as pd

def run_query(query: str, database: str = "master") -> pd.DataFrame:
    """
    All-in-one function to execute ANY T-SQL command (CREATE, INSERT, SELECT, etc.)
    against your local Microsoft SQL Server instance.
    """
    conn_str = (
        r"Driver={ODBC Driver 18 for SQL Server};"
        f"Server=DESKTOP-LVP84RD\MSSQLSERVER01;"
        f"Database={database};"
        r"Trusted_Connection=yes;"
        r"TrustServerCertificate=yes;"
    )
    
    conn = None
    try:
        # Open connection and enable autocommit so structural changes save instantly
        conn = pyodbc.connect(conn_str, autocommit=True)
        cursor = conn.cursor()
        
        # Execute the raw query string
        cursor.execute(query)
        
        # Check if the query returned rows (like a SELECT statement does)
        if cursor.description:
            # Fetch the metadata columns and rows into a structured DataFrame
            columns = [column[0] for column in cursor.description]
            rows = cursor.fetchall()
            # Convert pyodbc rows to standard python lists for Pandas compatibility
            data = [list(row) for row in rows]
            return pd.DataFrame(data, columns=columns)
        else:
            print("🚀 Command executed successfully (No rows returned).")
            return None
            
    except Exception as e:
        print(f"❌ Execution Error: {e}")
        return None
        
    finally:
        if conn:
            conn.close()

<>:11: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
<>:11: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
C:\Users\Nishad\AppData\Local\Temp\ipykernel_1908\2771818840.py:11: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
  f"Server=DESKTOP-LVP84RD\MSSQLSERVER01;"


In [9]:
print("--- 1. Creating the Database ---")
run_query("CREATE DATABASE LearnSQL;")

--- 1. Creating the Database ---
🚀 Command executed successfully (No rows returned).


In [10]:
print("\n--- 2. Creating the Table (Switching context to 'LearnSQL') ---")
create_table_sql = """
CREATE TABLE Employees (
    EmployeeID INT IDENTITY(1,1) PRIMARY KEY,
    FirstName VARCHAR(50) NOT NULL,
    LastName VARCHAR(50) NOT NULL,
    Department VARCHAR(50),
    Salary DECIMAL(10, 2),
    HireDate DATE
);
"""
run_query(create_table_sql, database="LearnSQL")


--- 2. Creating the Table (Switching context to 'LearnSQL') ---
🚀 Command executed successfully (No rows returned).


In [11]:
print("\n--- 3. Inserting Sample Entries ---")
insert_data_sql = """
INSERT INTO Employees (FirstName, LastName, Department, Salary, HireDate) VALUES
('Nishad', 'Sharma', 'Engineering', 95000.00, '2024-03-15'),
('Priya', 'Patel', 'Data Science', 105000.00, '2023-11-01'),
('Amit', 'Kumar', 'Engineering', 88000.00, '2025-01-10'),
('Sneha', 'Reddy', 'Marketing', 72000.00, '2024-07-22'),
('Rohan', 'Das', 'Data Science', 112000.00, '2022-05-18');
"""
run_query(insert_data_sql, database="LearnSQL")


--- 3. Inserting Sample Entries ---
🚀 Command executed successfully (No rows returned).


In [12]:
print("\n--- 4. Executing basic query (Retrieving everything) ---")
df_all = run_query("SELECT * FROM Employees;", database="LearnSQL")


--- 4. Executing basic query (Retrieving everything) ---


In [3]:
# Query the system database catalog
databases_df = run_query("SELECT name AS DatabaseName, create_date FROM sys.databases;")

# Display the table
databases_df


,DatabaseName,create_date
0,master,2003-04-08 09:13:36.390
1,tempdb,2026-11-09 20:15:34.737
2,model,2003-04-08 09:13:36.390
3,msdb,2025-10-21 12:53:13.233
4,LearnSQL,2026-09-21 18:21:46.770


In [15]:
# Query the system table catalog for your specific database
tables_df = run_query("SELECT name AS TableName, create_date FROM sys.tables;", database="LearnSQL")

# Display the table
tables_df


,TableName,create_date
0,Employees,2026-09-21 18:21:54.900


In [17]:
query_all = "SELECT * FROM Employees;"
run_query(query_all, database="LearnSQL")


,EmployeeID,FirstName,LastName,Department,Salary,HireDate
0,1,Nishad,Sharma,Engineering,95000.00,2024-03-15
1,2,Priya,Patel,Data Science,105000.00,2023-11-01
2,3,Amit,Kumar,Engineering,88000.00,2025-01-10
3,4,Sneha,Reddy,Marketing,72000.00,2024-07-22
4,5,Rohan,Das,Data Science,112000.00,2022-05-18


In [18]:
query_filter = """
SELECT FirstName, LastName, Salary 
FROM Employees 
WHERE Department = 'Engineering' AND Salary > 90000;
"""
run_query(query_filter, database="LearnSQL")


,FirstName,LastName,Salary
0,Nishad,Sharma,95000.00
